In [ ]:
import numpy as np
import pandas as pd

def build_features_from_raw(df_raw: pd.DataFrame) -> pd.DataFrame:
    d = df_raw.copy()

    # ---- 숫자형 변환 ----
    num_cols = ["Exercise_Duration", "Body_Temperature(F)", "BPM",
                "Height(Feet)", "Height(Remainder_Inches)", "Weight(lb)", "Age"]
    for c in num_cols:
        d[c] = pd.to_numeric(d[c], errors="coerce")

    # ---- 단위 변환 ----
    d["height_in"] = d["Height(Feet)"] * 12 + d["Height(Remainder_Inches)"]
    d["height_cm"] = d["height_in"] * 2.54

    d["weight_kg"] = d["Weight(lb)"] * 0.45359237
    d["temp_c"] = (d["Body_Temperature(F)"] - 32) * (5/9)

    # ---- BMI ----
    height_m = d["height_cm"] / 100.0
    d["bmi"] = d["weight_kg"] / (height_m ** 2)

    # ---- Gender 인코딩: F=0, M=1  ----
    d["Gender"] = d["Gender"].map({"F": 0, "M": 1}).astype("Int64")

    # ---- BMR (Mifflin-St Jeor) ----
    # 남: +5, 여: -161
    s = np.where(d["Gender"] == 1, 5, -161)
    d["bmr"] = 10*d["weight_kg"] + 6.25*d["height_cm"] - 5*d["Age"] + s

    # ---- MHR / HRR / HRR ratio ----
    d["mhr"] = 220 - d["Age"]
    rest_hr = 60
    d["hrr"] = d["mhr"] - rest_hr
    d["hrr_ratio"] = (d["BPM"] - rest_hr) / d["hrr"]

    # ---- activity_intensity ----
    # 간단 규칙: <100=0, 100~119=1, >=120=2
    d["activity_intensity"] = np.select(
        [d["BPM"] < 100, d["BPM"] < 120],
        [0, 1],
        default=2
    ).astype(int)

    # ---- thermal_load / hb_stress / metabolic_stress / relative_workload ----
    d["thermal_load"] = (d["temp_c"] - 37.0) * d["Exercise_Duration"]
    d["hb_stress"] = d["BPM"] / d["mhr"]
    d["metabolic_stress"] = d["Exercise_Duration"] * d["BPM"]
    d["relative_workload"] = d["metabolic_stress"] * d["weight_kg"] / d["Age"]

    # ---- Weight_Status 더미 ----
    d["Weight_Status_Overweight"] = (d["Weight_Status"] == "Overweight").astype(int)
    d["Weight_Status_Obese"] = (d["Weight_Status"] == "Obese").astype(int)

    # ---- 최종 컬럼 ----
    final_cols = [
        "Exercise_Duration","BPM","Gender","Age",
        "height_cm","weight_kg","temp_c","bmi","bmr","mhr",
        "activity_intensity","hrr","hrr_ratio","thermal_load","hb_stress",
        "metabolic_stress","relative_workload",
        "Weight_Status_Overweight","Weight_Status_Obese",
    ]
    return d[final_cols]


In [ ]:
import pandas as pd
from catboost import CatBoostRegressor

# 원본 로드
train_raw = pd.read_csv("train.csv")
test_raw  = pd.read_csv("test.csv")

# 파생변수 생성
X_train = build_features_from_raw(train_raw)
y_train = train_raw["Calories_Burned"]

X_test  = build_features_from_raw(test_raw)

# 최종 CatBoost (Optuna best)
model = CatBoostRegressor(
    iterations=2492,
    learning_rate=0.025309259727630214,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

# 제출 파일
submission = pd.DataFrame({
    "id": test_raw["ID"],                
    "Calories_Burned": pred
})
submission.to_csv("submission.csv", index=False)
print("saved:", submission.shape)
print(submission.head())


saved: (7500, 2)
          id  Calories_Burned
0  TEST_0000       173.556874
1  TEST_0001       189.421055
2  TEST_0002        53.190778
3  TEST_0003       160.899695
4  TEST_0004       225.157409
